# AI-QMS — Phase 2: Data Collection & Preprocessing

Dataset sources locked in for Phase 1: synthetic data now (blocks nothing), Google Form survey and public datasets to be reconciled into the same canonical schema once shared.

Canonical raw event schema (`pipeline/schema.py`): institution_id, counter_id, service_type, arrival_ts, called_ts, service_start_ts, service_end_ts, queue_length_at_arrival.

Pipeline stages (`pipeline/cleaning.py`): timestamp normalization (UTC) → missing-value policy → derived time metrics → outlier bounds → time features → categorical encoding.

In [ ]:
from pathlib import Path
from pipeline.generator import generate_events, write_raw
from pipeline.cleaning import run_from_file

raw_dir = Path("data/raw")
proc_dir = Path("data/processed")
raw_dir.mkdir(parents=True, exist_ok=True)
proc_dir.mkdir(parents=True, exist_ok=True)

raw = generate_events()
write_raw(raw, raw_dir / "queue_events.csv")
stats = run_from_file(str(raw_dir / "queue_events.csv"), str(proc_dir / "queue_events_clean.csv"))
stats

In [ ]:
import pandas as pd

df = pd.read_csv(proc_dir / "queue_events_clean.csv")
print("rows raw -> processed:", stats)
print("nulls after cleaning:", int(df.isna().sum().sum()))
print()
print(df[["wait_time_min", "service_time_min", "hour_of_day", "day_of_week", "is_weekend"]].describe().round(2).to_string())
print()
print("queue_length flagged missing:", int(df["queue_length_missing"].sum()))

## Design notes

- **Missing values:** rows missing any of arrival/called/service_start/service_end are dropped — they cannot yield a wait_time target (still-in-service). `queue_length_at_arrival` is flagged (`queue_length_missing`) and zero-filled rather than dropped.
- **Outliers:** wait_time_min ≤ 240, service_time_min ≤ 120, queue_length ≤ 200; winsorized (clipped) by default — switch `winsorize_bounds=False` to drop instead.
- **Time normalization:** all timestamps converted to UTC; features hour_of_day, day_of_week, is_weekend derived from arrival_ts.
- **Categorical encoding:** service_type one-hot (default) or label via `PipelineConfig.encode_categorical`.
- **Reconciliation:** when the Google Form responses and named public datasets arrive, they must be mapped onto this canonical schema before this pipeline runs; the mapping will be added here.